In [1]:
import polars as pl
import numpy as np
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import os
from collections import defaultdict
import datetime as dt
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mticker
sys.path.insert(1, os.path.abspath(".."))
os.chdir(os.path.abspath('..')) # Change the workdir to the parent folder

# ── Visual defaults ───────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
SIRS_COLORS = {0: "#4CAF50", 1: "#FFC107", 2: "#FF9800", 3: "#F44336", 4: "#9C27B0"}

In [2]:
from src.configs.dataconfig import input_output_config
from src.utils.utils import load_df

In [3]:
df_all = load_df(input_output_config.output_path, "df_all.parquet")
df_all_layer1 = load_df(input_output_config.output_path, "df_all_layer1.parquet")
df_sirs = load_df(input_output_config.output_path, "df_sirs.parquet")
df_sirs_filled = load_df(input_output_config.output_path, "df_sirs_filled.parquet")
df_agg = load_df(input_output_config.output_path, "df_agg.parquet")
df_organdysfunction = load_df(input_output_config.output_path, "df_organdysfunction.parquet")
df_sepsis1 = load_df(input_output_config.output_path, "df_sepsis1.parquet")
df_sepsis2 = load_df(input_output_config.output_path, "df_sepsis2.parquet")
df_sepsis3 = load_df(input_output_config.output_path, "df_sepsis3.parquet")
df_septicshock = load_df(input_output_config.output_path, "df_septicshock.parquet")
df_infect = load_df(input_output_config.output_path, "df_infect_long.parquet")

In [8]:
df_all.filter(
	pl.col("Event_Grouper") == "Vent On/Off"
)['Value'].unique().to_list()

['$ Home Vent Used',
 'On Going Hospital Vent',
 'Standby',
 'Initial',
 '$ On Going Hospital Vent',
 None]

In [12]:
df_all.filter(
    pl.col("Event_Name") == 'Shock, unspecified (*)'
)['Value'].unique().to_list()

['R57.9']

In [8]:
df_all['Event_Name'].value_counts(sort=True).to_pandas().head(100)

,Event_Name,count
0,PULSE,1561572
1,RESPIRATIONS,1326081
2,BLOOD PRESSURE,714273
3,UTSW R ARTERIAL BLOOD PRESSURE MEAN,541855
4,TEMPERATURE,371650
...,...,...
95,CULTURE SPUTUM + SCREENING SMEAR,2444
96,"Shock, unspecified (*)",2415
97,Hypo-osmolality and hyponatremia,2364
98,AZITHROMYCIN 500 MG IVPB (VIAL2BAG),2322


In [ ]:
list(filter(lambda x: 'O2' in x or 'vent' in x.lower(), sorted(df_all['Event_Grouper'].unique().to_list())))

['FIO2',
 'O2 Delivery High-Flow',
 'O2 Delivery Mechanical Ventilation',
 'O2 Delivery Nasal Cannula',
 'O2 Delivery Non-Rebreather Mask',
 'O2 Delivery Room Air',
 'O2 Delivery Simple Face Mask',
 'O2 Flow Rate',
 'PAO2',
 'Vent On/Off',
 'Vent off Documentation',
 'Vent on Documentation']

In [13]:
data_dict = {}
for c in ['O2 Delivery High-Flow',
 'O2 Delivery Mechanical Ventilation',
 'O2 Delivery Nasal Cannula',
 'O2 Delivery Non-Rebreather Mask',
 'O2 Delivery Room Air',
 'O2 Delivery Simple Face Mask',
 'O2 Flow Rate']:
	print(c)
	data_dict[c] = sorted(df_all.filter(
		pl.col("Event_Grouper") == c
	)['Value'].unique().to_list())

data_dict

O2 Delivery High-Flow
O2 Delivery Mechanical Ventilation
O2 Delivery Nasal Cannula
O2 Delivery Non-Rebreather Mask
O2 Delivery Room Air
O2 Delivery Simple Face Mask
O2 Flow Rate


TypeError: '<' not supported between instances of 'NoneType' and 'str'

In [17]:
df_all.filter(
    pl.col("Value").str.to_lowercase().str.contains("z93.0")
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN
i64,datetime[μs],str,str,str,f64,str,str,i64,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str
738207629,2025-10-14 02:01:59.587,"""Diagnosis Event""","""Hospital Problem""","""Tracheostomy dependent (*)""",null,"""Z93.0""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
741787392,2025-12-12 02:33:24.710,"""Diagnosis Event""","""Billing Diagnosis""","""Tracheostomy status (*)""",null,"""Z93.0""",null,99266466,741787392,2025-12-03 00:00:00,2025-12-23 00:00:00,2025-12-03 19:47:00,2025-12-04 01:12:00,"""Yes""","""Inpatient""","""Inpatient""","""ENT""","""20""","""UH 02O SICU""","""UH 06G""","""Emergency""","""Trans from Short Term General …","""OR Admission""","""Parapharyngeal abscess""","""Sepsis, unspecified organism""","""Inspection of Larynx, Endo""",0,"""POA-3""","""POA-3""",153.0,null,null,null,null,null,null,null,"""2025-12-03 19:56:00.0000000""","""Suspected Infection Flowsheet""","""0""","""0""","""0""","""0""","""0""","""0"""
732317863,2025-08-26 02:05:39.813,"""Diagnosis Event""","""Billing Diagnosis""","""Tracheostomy status (*)""",null,"""Z93.0""",null,90566124,732317863,2025-07-29 00:00:00,2025-08-15 00:00:00,2025-07-29 14:17:00,2025-07-29 20:31:00,"""Yes""","""Inpatient""","""Inpatient""","""MICU""","""17""","""UH 06O""","""UH 08B MICU""","""Emergency""","""Home & Outside Location""","""ED Admission""","""Seizure (*)""","""Infection and inflammatory rea…","""Assistance with Respiratory Ve…",0,"""POA-1""","""POA-1""",137.0,14.0,99.0,0.633333,214.5,null,null,null,"""2025-07-29 14:25:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""0""","""0""","""0""","""0"""
729590295,2025-07-03 01:58:52.630,"""Diagnosis Event""","""Problem List""","""Status post tracheostomy (*)""",null,"""Z93.0""",null,72867474,729590295,2025-06-20 00:00:00,2025-07-11 00:00:00,2025-06-20 04:31:00,2025-06-20 04:31:00,"""No""","""Inpatient""","""Inpatient""","""MICU""","""21""","""UH 03O NEURO ICU""","""UH 07B ICU""","""Urgent""","""Trans from Short Term General …","""Transfer Center Admission""","""Acute on chronic respiratory f…","""Pneumonitis due to inhalation …","""Respiratory Ventilation, Great…",0,"""NPOA-3""","""NPOA-3""",134.0,16.0,85.0,0.274583,296.6,0.31,134.714286,5.617,"""2025-07-03 12:00:00.0000000""","""IV Antibiotics + Blood Culture…","""0""","""0""","""1""","""0""","""0""","""0"""
729590295,2025-07-03 01:58:52.630,"""Diagnosis Event""","""Hospital Problem""","""Status post tracheostomy (*)""",null,"""Z93.0""",null,72867474,729590295,2025-06-20 00:00:00,2025-07-11 00:00:00,2025-06-20 04:31:00,2025-06-20 04:31:00,"""No""","""Inpatient""","""Inpatient""","""MICU""","""21""","""UH 03O NEURO ICU""","""UH 07B ICU""","""Urgent""","""Trans from Short Term General …","""Transfer Center Admission""","""Acute on chronic respiratory f…","""Pneumonitis due to inhalation …","""Respiratory Ventilation, Great…",0,"""NP

In [15]:
data_dict['O2 Delivery Nasal Cannula']

['aerosol mask',
 'blow-by',
 'nasal cannula',
 'nasal cannula with reservoir (Oximizer)',
 'open oxygen mask',
 'oxyhood',
 'tracheostomy collar',
 'transtracheal catheter']

In [16]:
df_all.filter( 
	pl.col("Event_Grouper") == "O2 Delivery Mechanical Ventilation"
).select(
    "EncounterEpicCsn",
	"Event_DateTime",
	"Type",
    "Event_Name",
    "Value",
    "NumericValue"
)['Value'].unique().to_list()

['T-piece',
 'CPAP',
 'mechanical ventilator',
 'nasal prongs',
 'ventilator',
 'manual resuscitator',
 'BiPAP',
 'NPPV/NIV',
 'T- piece']

### P/F Ratio Exploration

In [12]:
df_all.filter(
    pl.col("Event_Grouper") == "PAO2"
)['NumericValue'].unique()

NumericValue
f64
27.0
29.0
30.0
31.0
32.0
…
484.0
485.0
486.0


In [21]:
df_all.filter(
    pl.col("Event_Grouper") == "PAO2"
)['Value'].unique()
df_all.filter(
    pl.col("Event_Grouper") == "PAO2"
).with_columns(
    pl.col("Value").cast(pl.Int64, strict=False).alias('Value_int')
).filter(pl.col("Value").is_not_null() & pl.col("Value_int").is_null())['Value'].value_counts(sort=True)

Value,count
str,u64
""">488""",223
"""<30""",13


In [33]:
df_all.filter(
    pl.col("Event_Grouper") == "PAO2"
)['Value'].unique()
df_all.filter(
    pl.col("Event_Grouper") == "PAO2"
).with_columns(
    pl.col("Value").cast(pl.Int64, strict=False).alias('Value_int')
).with_columns(
    pl.when(
        pl.col("Value").is_not_null() & pl.col("Value_int").is_null() & 
		(pl.col("Value") == ">488")
	)
    .then(pl.lit(490)).otherwise(pl.col("Value_int")).alias("Value_int"),
).with_columns(
    pl.when(
        pl.col("Value").is_not_null() & pl.col("Value_int").is_null() & 
		(pl.col("Value") == "<30")
	)
    .then(pl.lit(29)).otherwise(pl.col("Value_int")).alias("Value_int")
).filter(
    pl.col("Value_int")!=pl.col("NumericValue")
).select(
    pl.col("NumericValue").value_counts(sort=True),
    pl.col("Value").value_counts(sort=True),
    pl.col("Value_int").value_counts(sort=True)
)


NumericValue,Value,Value_int
struct[2],struct[2],struct[2]
"{536.8,223}","{"">488"",223}","{490,223}"
"{27.0,13}","{""<30"",13}","{29,13}"


In [43]:
df_pao2 = df_all.filter(
    pl.col("Event_Grouper") == 'PAO2'
).select(
    "EncounterEpicCsn", "Event_DateTime", "Type", "Event_Name", "Value", "NumericValue"
)
df_fio2 = df_all.filter(
    pl.col("Event_Grouper") == 'FIO2'
).select(
    "EncounterEpicCsn", "Event_DateTime", "Type", "Event_Name", "Value", "NumericValue"
)
df_fio2.shape, df_pao2.shape

((24994, 6), (27875, 6))

In [45]:
df_pao2_fio2_joined = df_pao2.join(
    df_fio2,
    on='EncounterEpicCsn'
).filter(
    (pl.col("Event_DateTime")-pl.col("Event_DateTime_right")).abs() < pl.duration(hours=2)
).sort(by=['EncounterEpicCsn', 'Event_DateTime', 'Event_DateTime_right'])

In [49]:
df_pao2_fio2_joined.with_columns(
    pf_ratio = (pl.col("NumericValue")/(pl.col("NumericValue_right")/100.0))
).filter(
    pl.col("pf_ratio")<300
)['EncounterEpicCsn'].unique()

EncounterEpicCsn
i64
699476058
709516764
714445951
714534044
716208986
…
746523197
746580472
746717846


### O2 Delivery Exploration

In [52]:
df_all.filter(
    pl.col("Event_Grouper").str.to_lowercase().str.contains("o2")
)['Event_Grouper'].unique().to_list()

['O2 Delivery Non-Rebreather Mask',
 'O2 Delivery Room Air',
 'PAO2',
 'FIO2',
 'O2 Delivery Mechanical Ventilation',
 'O2 Delivery Nasal Cannula',
 'O2 Flow Rate',
 'O2 Delivery Simple Face Mask',
 'O2 Delivery High-Flow']

In [64]:
df_all.filter(
    pl.col("Event_Grouper") == "O2 Delivery Simple Face Mask"
)['Value'].unique().to_list()

['heated', 'humidified', 'face tent', 'simple face mask']

In [63]:
df_all.filter(
    (pl.col("Event_Grouper") == "O2 Delivery Simple Face Mask")&
    # (pl.col("Event_Name") == 'UTSW R ED PRE HOSPITAL CARE EMS VITALS INV O2 DEVICE (L/MIN)')
	(pl.col("NumericValue").is_not_null())
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,PrimaryMrn,EncounterEpicCsn_right,AdmissionDateValue,DischargeDateValue,Arrival_Instant,InpatientAdmissionInstant,Admitted_from_ED,PatientClass,InpatientAdmissionPatientClass,HospitalService,LengthOfStayInDays,AdmittingDepartment,DischargeDepartment,AdmissionType,AdmissionSource,AdmissionOrigin,PrincipalProblem,PrimaryCodedDiagnosis,PrimaryCodedProcedureKey,Death_Flag,Original_POA_Condition,Sepsis_Category,Baseline_SBP,Baseline_RespiratoryRate,Baseline_PulseRate,Baseline_Creatinine,Baseline_Platelets,Baseline_Bilirubin,Baseline_eGFR,Baseline_WBC,First_Suspected_Infection_Time,Suspected_Infection_Criteria_Met,Cancer_Registry_YN,HIV_Registry_YN,Immunocrompromised_Registry_YN,CKD_Dialysis_Registry_YN,Solid_Organ_Transplant_Registry_YN,Pregnancy_Registry_YN
i64,datetime[μs],str,str,str,f64,str,str,i64,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str
